LABORATORIO 5 - Similitud del coseno

Obtiene la similitud entre las consultas y los documentos de la colección CACM usando el coseno
sobre las matrices tf-idf (vocabulario reducido) del laboratorio 4.

Entradas (práctica 4): matriz_tfidf_documentos.npz, matriz_tfidf_consultas.npz, ids_documentos.txt e ids_consultas.txt.

Entrada (práctica 3): vocabularioReducido.txt (solo para mostrar los nombres de los términos en la verificación).

Salida (carpeta Practica5): CACM_tf_idf_rels.txt, con una línea por cada par consulta-documento cuyo coseno es distinto de 0:

    consulta documento coseno        (ejemplo: 001 1220 0.400000)

Dentro de cada consulta los documentos van ordenados de mayor a menor coseno.

In [1]:
import os

In [2]:
import numpy as np

In [3]:
from scipy import sparse

In [4]:
from scipy.sparse.linalg import norm

In [5]:
#CARPETAS
carpeta_vocabulario = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica3"
corpus_root = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica4"
ruta_salida = r"C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica5\CACM_tf_idf_rels.txt"

LECTURA DE LAS MATRICES TF-IDF

Se cargan las matrices guardadas en el laboratorio 4. Cada renglón es un documento (o una consulta) y cada columna
es un término del vocabulario reducido. Se usa el formato CSR para conservar la representación dispersa.

In [6]:
tfidf_docs = sparse.load_npz(os.path.join(corpus_root, "matriz_tfidf_documentos.npz")).tocsr()
tfidf_consultas = sparse.load_npz(os.path.join(corpus_root, "matriz_tfidf_consultas.npz")).tocsr()
print("Matriz tf-idf de documentos:", tfidf_docs.shape)
print("Matriz tf-idf de consultas:", tfidf_consultas.shape)

Matriz tf-idf de documentos: (3204, 3016)
Matriz tf-idf de consultas: (64, 3016)


LECTURA DE LOS IDENTIFICADORES

Los archivos ids_documentos.txt e ids_consultas.txt guardan el número de documento (o de consulta) de cada renglón
de las matrices, en el mismo orden. Sirven para escribir en el archivo final el número real y no la posición.

In [7]:
# un identificador por linea, en el mismo orden que los renglones de la matriz
def leer_ids(nombre_archivo):
    with open(os.path.join(corpus_root, nombre_archivo), encoding="utf-8") as f:
        return [linea.strip() for linea in f if linea.strip()]

In [8]:
ids_docs = leer_ids("ids_documentos.txt")
ids_consultas = leer_ids("ids_consultas.txt")
print(len(ids_docs), "identificadores de documentos | primeros:", ids_docs[:5])
print(len(ids_consultas), "identificadores de consultas | primeros:", ids_consultas[:5])

3204 identificadores de documentos | primeros: ['1', '2', '3', '4', '5']
64 identificadores de consultas | primeros: ['1', '2', '3', '4', '5']


VALIDACIONES DE INTEGRIDAD

Antes de calcular nada se comprueba que las matrices y los identificadores sean consistentes entre sí.
Si alguna validación falla, el programa se detiene con un mensaje.

In [9]:
if tfidf_docs.shape[0] != len(ids_docs):
    raise ValueError("La cantidad de documentos no coincide con ids_documentos.txt")
if tfidf_consultas.shape[0] != len(ids_consultas):
    raise ValueError("La cantidad de consultas no coincide con ids_consultas.txt")
if tfidf_docs.shape[1] != tfidf_consultas.shape[1]:
    raise ValueError("Documentos y consultas deben tener el mismo vocabulario")
print("Validaciones correctas. Dimension de los vectores:", tfidf_docs.shape[1])

Validaciones correctas. Dimension de los vectores: 3016


SIMILITUD DEL COSENO

coseno(q, d) = (q . d) / (||q|| * ||d||)

El numerador es el producto punto entre el vector tf-idf de la consulta y el del documento; el denominador es el
producto de sus normas L2 (longitudes). Se calcula en tres pasos para todas las consultas contra todos los documentos a la vez.

Paso 1: numerador. La multiplicación matricial consultas x documentos^T da en un solo paso todos los productos punto.

In [10]:
productos = (tfidf_consultas @ tfidf_docs.T).toarray()
print("Matriz de productos punto (consultas x documentos):", productos.shape)
print("Pares con producto punto distinto de 0:", np.count_nonzero(productos))

Matriz de productos punto (consultas x documentos): (64, 3204)
Pares con producto punto distinto de 0: 45339


Paso 2: denominador. Se calcula la norma L2 de cada renglón y se combinan todas las parejas consulta-documento
con una multiplicación de columna por renglón.

In [11]:
normas_consultas = norm(tfidf_consultas, axis=1)
normas_docs = norm(tfidf_docs, axis=1)
denominador = normas_consultas[:, None] * normas_docs[None, :]
print("Norma de la consulta 1: %.4f" % normas_consultas[0])
print("Norma del documento 1: %.4f" % normas_docs[0])
print("Consultas con norma 0:", int(np.sum(normas_consultas == 0)), "| documentos con norma 0:", int(np.sum(normas_docs == 0)))

Norma de la consulta 1: 4.1731
Norma del documento 1: 3.8079
Consultas con norma 0: 0 | documentos con norma 0: 33


Paso 3: división. Si algún vector es nulo (norma 0) el coseno se deja en 0 en lugar de dividir entre cero.
Con tf-idf todos los pesos son mayores o iguales a 0, así que el coseno debe quedar entre 0 y 1.

In [12]:
cosenos = np.zeros(productos.shape, dtype=float)
np.divide(productos, denominador, out=cosenos, where=denominador != 0)
print("Matriz de cosenos:", cosenos.shape)
print("Coseno minimo: %.4f | coseno maximo: %.4f" % (cosenos.min(), cosenos.max()))

Matriz de cosenos: (64, 3204)
Coseno minimo: 0.0000 | coseno maximo: 0.8164


FILTRADO Y ORDENAMIENTO POR CONSULTA

Para cada consulta se conservan solo los documentos con coseno distinto de 0 y se ordenan de mayor a menor.
El ordenamiento es estable: si dos documentos empatan, se respeta el orden en que aparecen en la colección.

In [13]:
# devuelve {consulta: [(documento, coseno), ...]} ordenado de mayor a menor coseno
def ordenar_resultados(cosenos, ids_consultas, ids_docs):
    resultados = {}
    for i, q_id in enumerate(ids_consultas):
        indices = np.flatnonzero(cosenos[i] != 0)                            # solo cosenos distintos de 0
        indices = indices[np.argsort(-cosenos[i, indices], kind="stable")]   # de mayor a menor
        resultados[q_id] = [(ids_docs[j], float(cosenos[i, j])) for j in indices]
    return resultados

In [14]:
resultados = ordenar_resultados(cosenos, ids_consultas, ids_docs)
print("Pares con coseno diferente de cero:", sum(len(v) for v in resultados.values()))
for q_id in ids_consultas[:3]:
    print("Consulta %s: %d documentos relevantes" % (q_id, len(resultados[q_id])))
    for d_id, similitud in resultados[q_id][:3]:
        print("  Documento %s: similitud = %.4f" % (d_id, similitud))

Pares con coseno diferente de cero: 45339
Consulta 1: 798 documentos relevantes
  Documento 1938: similitud = 0.2951
  Documento 31: similitud = 0.2806
  Documento 1071: similitud = 0.2417
Consulta 2: 192 documentos relevantes
  Documento 356: similitud = 0.4647
  Documento 740: similitud = 0.4647
  Documento 136: similitud = 0.2894
Consulta 3: 567 documentos relevantes
  Documento 1521: similitud = 0.2699
  Documento 1134: similitud = 0.2498
  Documento 1304: similitud = 0.2262


GENERACIÓN DEL ARCHIVO DE SALIDA

Cada línea es: número de consulta (con tres dígitos, como 001), número de documento y coseno.
El coseno se escribe con 6 decimales para que ningún valor distinto de 0 se vea como 0.000000 en el archivo.

In [15]:
def guardar_resultados(resultados, ruta, decimales=6):
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    with open(ruta, "w", encoding="utf-8") as salida:
        for q_id, lista in resultados.items():
            for d_id, similitud in lista:
                salida.write(f"{q_id.zfill(3)} {d_id} {similitud:.{decimales}f}\n")
    print("Archivo generado:", ruta)

In [16]:
guardar_resultados(resultados, ruta_salida)

Archivo generado: C:\Users\erick\OneDrive\Documentos\documentos UNI\Semestre 8\Recuperacion de info\Practicas en clase\Practica5\CACM_tf_idf_rels.txt


VERIFICACION (para la sección de resultados del reporte)

Se lee de nuevo el archivo generado para comprobar su formato, y después se recalcula a mano el coseno de un par
consulta-documento para confirmar que coincide con el valor del programa.

In [17]:
# primeras lineas del archivo guardado
with open(ruta_salida, encoding="utf-8") as f:
    lineas = [linea.split() for linea in f.read().splitlines()]
for campos in lineas[:5]:
    print(" ".join(campos))

001 1938 0.295090
001 31 0.280639
001 1071 0.241657
001 379 0.236240
001 1572 0.224975


In [18]:
total_pares = sum(len(v) for v in resultados.values())
print("Lineas en el archivo:", len(lineas), "| pares calculados:", total_pares, "| coinciden:", len(lineas) == total_pares)
print("Todas las lineas tienen 3 campos:", all(len(campos) == 3 for campos in lineas))
print("Lineas con similitud igual a 0:", sum(1 for campos in lineas if float(campos[2]) == 0))

# el orden descendente se revisa dentro de cada consulta
por_consulta = {}
for q, d, s in lineas:
    por_consulta.setdefault(q, []).append(float(s))
print("Consultas con resultados:", len(por_consulta), "| orden descendente en todas:",
      all(v == sorted(v, reverse=True) for v in por_consulta.values()))
print("Consultas sin ningun documento:", [q.zfill(3) for q in ids_consultas if not resultados[q]])

Lineas en el archivo: 45339 | pares calculados: 45339 | coinciden: True
Todas las lineas tienen 3 campos: True
Lineas con similitud igual a 0: 0
Consultas con resultados: 64 | orden descendente en todas: True
Consultas sin ningun documento: []


Cálculo a mano de un par consulta-documento. Se toma la consulta 1 y el documento con mayor similitud para ella,
y se muestran los términos que comparten, porque solo esos términos aportan al producto punto.

In [19]:
# vocabulario de la practica 3, solo para mostrar los nombres de los terminos
with open(os.path.join(carpeta_vocabulario, "vocabularioReducido.txt"), encoding="utf-8") as f:
    vocabulario_reducido = sorted(f.read().split())
print("Terminos en el vocabulario:", len(vocabulario_reducido), "| columnas de las matrices:", tfidf_docs.shape[1])

Terminos en el vocabulario: 3016 | columnas de las matrices: 3016


In [20]:
num_q = "1"
q = ids_consultas.index(num_q)
num_doc = resultados[num_q][0][0]      # documento con mayor similitud para la consulta
d = ids_docs.index(num_doc)

vector_q = tfidf_consultas[q].toarray().ravel()
vector_d = tfidf_docs[d].toarray().ravel()

print("Consulta", num_q, "(renglon", q, ") contra documento", num_doc, "(renglon", d, ")")
print("Terminos compartidos:")
for k in np.nonzero(vector_q * vector_d)[0]:
    print("  %5d  %-12s  q = %.4f  d = %.4f  q*d = %.4f" % (k, vocabulario_reducido[k], vector_q[k], vector_d[k], vector_q[k] * vector_d[k]))

Consulta 1 (renglon 0 ) contra documento 1938 (renglon 1937 )
Terminos compartidos:
    162  articl        q = 2.2047  d = 2.2047  q*d = 4.8605
   2425  share         q = 1.5145  d = 7.5723  q*d = 11.4680
   2748  time          q = 0.8908  d = 4.4540  q*d = 3.9676


In [21]:
producto = np.dot(vector_q, vector_d)
norma_q = np.linalg.norm(vector_q)
norma_d = np.linalg.norm(vector_d)
coseno_manual = producto / (norma_q * norma_d)

print("Producto punto = %.4f | ||q|| = %.4f | ||d|| = %.4f" % (producto, norma_q, norma_d))
print("Coseno a mano = %.6f | coseno del programa = %.6f" % (coseno_manual, cosenos[q, d]))
print("Coinciden:", np.isclose(coseno_manual, cosenos[q, d]))

Producto punto = 20.2962 | ||q|| = 4.1731 | ||d|| = 16.4814
Coseno a mano = 0.295090 | coseno del programa = 0.295090
Coinciden: True


In [22]:
# un documento sin terminos en comun con la consulta tiene coseno 0 y no debe aparecer en el archivo
ceros = np.flatnonzero(cosenos[q] == 0)
if len(ceros) > 0:
    sin_relacion = ids_docs[ceros[0]]
    print("Documento", sin_relacion, "-> coseno con la consulta", num_q, "=", cosenos[q, ceros[0]])
    print("Aparece en el resultado de la consulta:", sin_relacion in [d_id for d_id, _ in resultados[num_q]])

Documento 1 -> coseno con la consulta 1 = 0.0
Aparece en el resultado de la consulta: False
